<a href="https://colab.research.google.com/github/3430-dotcom/KOAI/blob/main/ai%EB%A5%BC_%ED%95%98%EC%9E%90~.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

layer

In [ ]:
import tensorflow as tf

class Layer(tf.Module):
    """Keras 레이어를 위한 기본 클래스입니다.

    레이어는 하나 이상의 텐서를 입력으로 받아 하나 이상의 텐서를 출력하는 호출 가능한 객체입니다.
    여기에는 입력 처리, 텐서 변환이 포함되며, 일반적으로 훈련 중에 학습되는 구성 가능한 가중치를 가집니다.

    속성:
        name: 레이어의 이름입니다. 제공되지 않으면 자동으로 생성됩니다.
        dtype: 레이어 계산의 dtype입니다. 기본값은 `tf.float32`입니다.
        trainable: 불리언, 레이어의 변수가 훈련 중에 업데이트되어야 하는지 여부입니다.
        built: 불리언, 레이어가 빌드되었는지 여부입니다.
    """

    def __init__(
        self,
        trainable=True,
        name=None,
        dtype=None,
        dynamic=False,
        **kwargs
    ):
        super().__init__(name=name)
        self._trainable = trainable
        self._dynamic = dynamic
        self._compute_dtype = tf.as_dtype(dtype or tf.keras.backend.floatx())
        self._compute_dtype_object = tf.dtypes.as_dtype(self._compute_dtype)
        self.built = False
        self._is_enable_get_config = True
        self._autocast = tf.keras.mixed_precision.autocast_scope()
        self._input_spec = None # Keras InputSpec은 일반적으로 여기에 사용됩니다.
        self._activity_regularizer = None # Activity regularizers는 이 단순화된 클래스에서 완전히 구현되지 않았습니다.

        # 이 레이어에 속한 `tf.Variable` 객체 목록입니다.
        self._trainable_weights = []
        self._non_trainable_weights = []
        self._updates = [] # 레거시 Keras 1.x 업데이트
        self._losses = [] # 정규화 손실

    def add_weight(
        self,
        name,
        shape,
        dtype=None,
        initializer=None,
        regularizer=None,
        trainable=True,
        constraint=None,
        **kwargs
    ):
        """레이어에 새 가중치 변수를 추가합니다.

        인자:
            name: 새 가중치의 이름입니다.
            shape: 정수 튜플 또는 리스트, 가중치의 형태입니다.
            dtype: 가중치의 데이터 타입입니다.
            initializer: 가중치를 위한 초기화 함수입니다.
            regularizer: 가중치를 위한 정규화 함수입니다.
            trainable: 불리언, 가중치가 훈련 가능해야 하는지 여부입니다.
            constraint: 가중치를 위한 제약 함수입니다.

        반환:
            `tf.Variable` 인스턴스입니다.
        """
        if dtype is None:
            dtype = self._compute_dtype
        if initializer is None:
            initializer = tf.keras.initializers.glorot_uniform()

        variable = tf.Variable(
            initializer(shape, dtype=dtype),
            dtype=dtype,
            name=name,
            trainable=trainable,
            constraint=constraint,
        )

        if trainable:
            self._trainable_weights.append(variable)
        else:
            self._non_trainable_weights.append(variable)

        if regularizer is not None:
            self._losses.append(regularizer(variable))

        return variable

    @property
    def weights(self):
        return self.trainable_weights + self.non_trainable_weights

    @property
    def trainable_weights(self):
        return self._trainable_weights if self.trainable else []

    @property
    def non_trainable_weights(self):
        # Non-trainable weights는 self.trainable 여부와 관계없이 항상 반환됩니다.
        if not self.trainable:
            return self._trainable_weights + self._non_trainable_weights
        return self._non_trainable_weights

    @property
    def trainable(self):
        return self._trainable

    @trainable.setter
    def trainable(self, value):
        self._trainable = value

    def build(self, input_shape):
        """레이어의 가중치를 생성합니다. `__call__`에서 한 번 호출됩니다.

        인자:
            input_shape: 레이어 입력의 형태(들)를 나타내는 `tf.TensorShape` 또는 `tf.TensorShape`s의 튜플/리스트입니다.
        """
        self.built = True

    def __call__(self, inputs, *args, **kwargs):
        """레이어의 주요 진입점입니다.

        인자:
            inputs: 입력 텐서(들).
            *args: `call()`로 전달되는 위치 인자입니다.
            **kwargs: `call()`로 전달되는 키워드 인자입니다.

        반환:
            레이어의 출력 텐서(들).
        """
        if not self.built:
            self.build(tf.shape(inputs))

        # 혼합 정밀도 자동 캐스팅이 활성화된 경우 적용
        with self._autocast:
            outputs = self.call(inputs, *args, **kwargs)
        return outputs

    def call(self, inputs, *args, **kwargs):
        """레이어의 로직이 여기에 있습니다.

        인자:
            inputs: 입력 텐서(들).
            *args: 위치 인자.
            **kwargs: 키워드 인자.

        반환:
            출력 텐서(들).
        """
        raise NotImplementedError("레이어는 자체 `call()` 메서드를 구현해야 합니다.")

    def compute_output_shape(self, input_shape):
        """레이어의 출력 형태를 계산합니다.

        레이어가 여러 입력을 가지는 경우, `input_shape`는 형태 리스트가 됩니다.
        레이어가 여러 출력을 가지는 경우, 이 메서드는 형태 리스트를 반환해야 합니다.

        인자:
            input_shape: `tf.TensorShape` 또는 `tf.TensorShape`s의 리스트/튜플입니다.

        반환:
            `tf.TensorShape` 또는 `tf.TensorShape`s의 리스트/튜플입니다.
        """
        # 기본적으로 단순 레이어의 경우 출력 형태가 입력 형태와 동일하다고 가정합니다.
        # 출력 형태가 변경되면 서브클래스는 이 메서드를 오버라이드해야 합니다.
        return input_shape

    def get_config(self):
        """레이어의 구성을 반환합니다.

        레이어의 구성은 레이어의 구성을 포함하는 Python 딕셔너리(직렬화 가능)입니다.
        동일한 레이어는 이 구성에서 나중에 `from_config()`를 통해 다시 인스턴스화될 수 있습니다.

        반환:
            레이어의 구성을 포함하는 Python 딕셔너리입니다.
        """
        config = {
            "name": self.name,
            "trainable": self.trainable,
            "dtype": self.dtype_policy.name if hasattr(self, 'dtype_policy') else tf.dtypes.as_dtype(self.dtype).name,
            "dynamic": self.dynamic,
        }
        return config

    @classmethod
    def from_config(cls, config):
        """구성으로부터 레이어를 생성합니다.

        이 메서드는 `get_config()`의 역으로, `get_config()`의 출력 구성으로부터 동일한 레이어 인스턴스를 재생성할 수 있습니다.

        인자:
            config: 일반적으로 `get_config()`의 출력인 Python 딕셔너리입니다.

        반환:
            레이어 인스턴스입니다.
        """
        return cls(**config)

    @property
    def dtype(self):
        """레이어 계산 및 가중치의 dtype입니다.

        `None`인 경우, 기본 `tf.keras.backend.floatx()`가 사용됩니다.
        """
        return self._compute_dtype

    @property
    def dynamic(self):
        """레이어가 동적인지 여부입니다 (런타임에 변하는 입력 형태를 지원).

        동적 레이어는 `build()` 중에 가중치를 생성하지 않습니다.
        """
        return self._dynamic

dense

In [ ]:
from tensorflow.keras.layers import Layer
from tensorflow.keras import activations, initializers, regularizers, constraints
import tensorflow as tf

class Dense(Layer):
    """일반적인 조밀하게 연결된 NN 레이어입니다.

    `Dense`는 다음 연산을 구현합니다:
    `output = activation(dot(input, kernel) + bias)`
    여기서 `activation`은 `activation` 인자로 전달된 요소별 활성화 함수이고,
    `kernel`은 레이어에 의해 생성된 가중치 행렬이며,
    `bias`는 레이어에 의해 생성된 바이어스 벡터입니다 (오직 `use_bias`가 `True`인 경우).

    참고: 레이어에 대한 입력의 랭크가 2보다 큰 경우, `Dense`는
    입력의 마지막 축과 `kernel`을 따라 내적을 계산하고
    입력의 나머지 축을 평탄화합니다.
    예: 입력 형태가 `(batch_size, d0, d1)`인 경우,
    출력 형태는 `(batch_size, d0, units)`이 됩니다.

    인자:
        units: 양의 정수, 출력 공간의 차원입니다.
        activation: 사용할 활성화 함수입니다.
            아무것도 지정하지 않으면 활성화가 적용되지 않습니다
            (즉, "선형" 활성화: `a(x) = x`).
        use_bias: 불리언, 레이어가 바이어스 벡터를 사용하는지 여부입니다.
        kernel_initializer: `kernel` 가중치 행렬의 초기화기입니다.
        bias_initializer: 바이어스 벡터의 초기화기입니다.
        kernel_regularizer: `kernel` 가중치 행렬에 적용되는 정규화 함수입니다.
        bias_regularizer: 바이어스 벡터에 적용되는 정규화 함수입니다.
        activity_regularizer: 레이어의 출력("활성화")에 적용되는 정규화 함수입니다.
        kernel_constraint: `kernel` 가중치 행렬에 적용되는 제약 함수입니다.
        bias_constraint: 바이어스 벡터에 적용되는 제약 함수입니다.

    입력 형태:
        형태가 `(batch_size, ..., input_dim)`인 N-D 텐서입니다.
        가장 일반적인 상황은 형태가 `(batch_size, input_dim)`인 2D 입력일 것입니다.

    출력 형태:
        형태가 `(batch_size, ..., units)`인 N-D 텐서입니다.
        예를 들어, 형태가 `(batch_size, input_dim)`인 2D 입력의 경우,
        출력 형태는 `(batch_size, units)`이 됩니다.
    """

    def __init__(
        self,
        units,
        activation=None,
        use_bias=True,
        kernel_initializer="glorot_uniform",
        bias_initializer="zeros",
        kernel_regularizer=None,
        bias_regularizer=None,
        activity_regularizer=None,
        kernel_constraint=None,
        bias_constraint=None,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.units = int(units)
        self.activation = activations.get(activation)
        self.use_bias = use_bias
        self.kernel_initializer = initializers.get(kernel_initializer)
        self.bias_initializer = initializers.get(bias_initializer)
        self.kernel_regularizer = regularizers.get(kernel_regularizer)
        self.bias_regularizer = regularizers.get(bias_regularizer)
        self.activity_regularizer = regularizers.get(activity_regularizer)
        self.kernel_constraint = constraints.get(kernel_constraint)
        self.bias_constraint = constraints.get(bias_constraint)

        # Keras `InputSpec`는 이 컨텍스트에서 일반적인 사용을 위해 직접 노출되지 않습니다.
        # 완전한 기능적 구현을 위해서는 `InputSpec`를 가져오거나 다시 생성해야 합니다.
        # self.input_spec = InputSpec(min_ndim=2)
        self.input_spec = None # InputSpec이 가져오거나 정의되지 않은 경우의 플레이스홀더

    def build(self, input_shape):
        # input_shape가 TensorShape 또는 호환되는지 확인합니다.
        if isinstance(input_shape, (list, tuple)):
            input_shape = tf.TensorShape(input_shape)

        input_dim = input_shape[-1]
        self.kernel = self.add_weight(
            "kernel",
            shape=[input_dim, self.units],
            initializer=self.kernel_initializer,
            regularizer=self.kernel_regularizer,
            constraint=self.kernel_constraint,
            dtype=self.dtype,
            trainable=True,
        )
        if self.use_bias:
            self.bias = self.add_weight(
                "bias",
                shape=[self.units,],
                initializer=self.bias_initializer,
                regularizer=self.bias_regularizer,
                constraint=self.bias_constraint,
                dtype=self.dtype,
                trainable=True,
            )
        else:
            self.bias = None
        self.built = True

    def call(self, inputs):
        if inputs.dtype.base_dtype != self._compute_dtype_object.base_dtype:
            inputs = tf.cast(inputs, dtype=self._compute_dtype_object)

        # 랭크 > 2인 입력을 암묵적으로 재구성하여 처리합니다.
        # tf.tensordot은 축이 올바르게 지정되고 input_dim이 입력의 마지막 차원인 경우 기본적으로 이를 처리합니다.
        outputs = tf.tensordot(inputs, self.kernel, axes=[[inputs.shape.rank - 1], [0]])

        if self.bias is not None:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return outputs

    def compute_output_shape(self, input_shape):
        if isinstance(input_shape, (list, tuple)):
            input_shape = tf.TensorShape(input_shape)

        output_shape = list(input_shape)
        output_shape[-1] = self.units
        return tf.TensorShape(output_shape)

    def get_config(self):
        config = super().get_config()
        config.update({
            "units": self.units,
            "activation": activations.serialize(self.activation),
            "use_bias": self.use_bias,
            "kernel_initializer": initializers.serialize(self.kernel_initializer),
            "bias_initializer": initializers.serialize(self.bias_initializer),
            "kernel_regularizer": regularizers.serialize(self.kernel_regularizer),
            "bias_regularizer": regularizers.serialize(self.bias_regularizer),
            "activity_regularizer": regularizers.serialize(self.activity_regularizer),
            "kernel_constraint": constraints.serialize(self.kernel_constraint),
            "bias_constraint": constraints.serialize(self.bias_constraint),
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

rnn

In [ ]:
from tensorflow.keras.layers import Layer
from tensorflow.keras import activations, initializers, regularizers, constraints
import tensorflow as tf

# 참고: 원본 Keras RNN 클래스는 `tf.keras.layers.StackedRNNCells` 및
# `tf.keras.utils.serialize_keras_object`/`deserialize_keras_object`에 의존합니다.
# Keras의 전체 내부 컨텍스트 없이 노트북에서 직접 실행하려면,
# `InputSpec`도 정의하거나 모의(mock)해야 합니다.
# 단순화를 위해, 이 버전은 핵심 구조에 중점을 두며
# 완전한 Keras와 같은 기능을 위해서는 추가 조정이 필요할 수 있습니다.

class RNN(Layer):
    """순환 레이어를 위한 기본 클래스입니다.

    인자:
        cell: RNN 셀 인스턴스 또는 RNN 셀 인스턴스 목록입니다.
            RNN 셀은 다음을 가지는 클래스입니다:
            - `call(self, inputs, states)` 메서드, 여기서 `inputs`는
              현재 타임스텝의 2D 입력이고, `states`는 2D 이전 상태 목록입니다.
            - `state_size` 속성, 이는 각 상태의 크기에 해당하는 정수 목록(또는 단일 정수)입니다.
            - `output_size` 속성입니다.
            - `get_initial_state(self, inputs=None, batch_size=None,
              dtype=None)` 메서드, 이는 초기 상태 텐서(들)를 생성합니다.
        return_sequences: 불리언. 출력 시퀀스의 마지막 출력을 반환할지,
            아니면 전체 시퀀스를 반환할지 여부입니다.
        return_state: 불리언. 출력 외에 마지막 상태를 반환할지 여부입니다.
        go_backwards: 불리언 (기본값 `False`). True인 경우, 입력 시퀀스를
            역방향으로 처리하고 역순 시퀀스를 반환합니다.
        stateful: 불리언 (기본값 `False`). True인 경우, 배치 내 인덱스 `i`의
            각 샘플에 대한 마지막 상태는 다음 배치 내 인덱스 `i`의 샘플에 대한
            초기 상태로 사용됩니다.
        unroll: 불리언 (기본값 `False`). True인 경우, 네트워크는 펼쳐지며,
            그렇지 않으면 기호 루프가 사용됩니다. `input_shape`가 정적이고 작을 때
            펼치기는 RNN의 속도를 높일 수 있습니다.
        time_major: 불리언 (기본값 `False`). True인 경우, 입력 및 출력은
            `(timesteps, batch, feature)` 형태가 되며, `False` (기본값) 옵션은
            `(batch, timesteps, feature)`를 사용합니다.
    """

    def __init__(
        self,
        cell,
        return_sequences=False,
        return_state=False,
        go_backwards=False,
        stateful=False,
        unroll=False,
        time_major=False,
        **kwargs,
    ):
        super().__init__(**kwargs)
        # cell이 목록인 경우, 일반적으로 StackedRNNCells로 래핑됩니다.
        # 이 독립 실행형 코드의 경우, `cell`이 단일 RNN 셀 인스턴스라고 가정합니다.
        if isinstance(cell, (list, tuple)):
            raise ValueError("이 단순화된 독립 실행형 RNN 클래스에서는 셀 목록이 지원되지 않습니다. 단일 셀 인스턴스를 제공하세요.")

        self.cell = cell
        self.return_sequences = return_sequences
        self.return_state = return_state
        self.go_backwards = go_backwards
        self.stateful = stateful
        self.unroll = unroll
        self.time_major = time_major

        # self.input_spec = [InputSpec(ndim=3)] # Keras InputSpec은 여기에 가져오지 않았습니다.
        self.supports_masking = True

        self._compute_output_and_states = None

    def build(self, input_shape):
        if isinstance(input_shape, list):
            input_shape = input_shape[0]

        # 완전한 Keras 컨텍스트에서는 InputSpec가 여기에 사용됩니다.
        # input_spec = InputSpec(shape=[None, None, input_shape[-1]])
        # self.cell.build(input_spec.shape)
        # 이 예시에서는 셀과 관련된 input_shape를 직접 전달합니다.
        self.cell.build(input_shape[-1]) # 셀이 입력의 마지막 차원에 따라 빌드된다고 가정합니다.
        self.built = True

    def call(self, inputs, initial_state=None, training=None, mask=None):
        # inputs 형태: (batch_size, timesteps, input_dim) 또는 (timesteps, batch_size, input_dim)
        # initial_state 형태: (batch_size, state_size_i) 텐서 목록

        if self.time_major:
            # (timesteps, batch_size, input_dim) -> (batch_size, timesteps, input_dim) 쉬운 처리를 위해
            inputs = tf.transpose(inputs, perm=(1, 0, 2))
            if mask is not None:
                mask = tf.transpose(mask, perm=(1, 0))

        if initial_state is None:
            if self.stateful:
                # 완전한 Keras RNN에서는 이전 배치의 상태를 검색합니다.
                # 이 단순화된 버전에서는 외부에서 제공되거나
                # 새로운 초기 상태를 얻어야 한다고 가정합니다.
                initial_state = self.cell.get_initial_state(
                    inputs=inputs, batch_size=tf.shape(inputs)[0], dtype=self.dtype)
            else:
                initial_state = self.cell.get_initial_state(
                    inputs=inputs, batch_size=tf.shape(inputs)[0], dtype=self.dtype)

        # `_run_rnn_loop`는 Keras의 복잡한 내부 메서드입니다.
        # 여기서는 시연을 위해 단순화된 직접 루프를 제공합니다.

        batch_size = tf.shape(inputs)[0]
        timesteps = tf.shape(inputs)[1]

        last_output = None
        outputs_list = []
        current_states = initial_state

        # go_backwards 처리
        if self.go_backwards:
            inputs = tf.reverse(inputs, axis=[1])
            if mask is not None:
                mask = tf.reverse(mask, axis=[1])

        for t in tf.range(timesteps):
            current_input = inputs[:, t, :]
            current_mask = mask[:, t] if mask is not None else None

            # 셀 호출 적용
            cell_output_and_states = self.cell(current_input, current_states, training=training)

            if isinstance(cell_output_and_states, (list, tuple)):
                current_output = cell_output_and_states[0]
                next_states = cell_output_and_states[1:]
            else:
                current_output = cell_output_and_states
                # 셀이 출력만 반환하는 경우, 상태는 출력 자체라고 가정합니다.
                next_states = [current_output] # 매우 단순화된 것으로, 실제 RNN 셀에는 일반적이지 않습니다.

            # 마스킹이 제공된 경우 적용
            if current_mask is not None:
                current_mask_expanded = tf.expand_dims(tf.cast(current_mask, dtype=current_output.dtype), axis=-1)
                current_output = current_output * current_mask_expanded
                next_states = [state * current_mask_expanded for state in next_states]

            outputs_list.append(current_output)
            current_states = next_states

        # 최종 출력 처리
        if self.return_sequences:
            outputs = tf.stack(outputs_list, axis=1) # (batch, timesteps, output_dim)
            if self.go_backwards:
                outputs = tf.reverse(outputs, axis=[1])
        else:
            outputs = outputs_list[-1] # 시퀀스의 마지막 출력

        last_output = outputs_list[-1]

        if self.return_state:
            return [outputs] + current_states
        return outputs

    def compute_output_shape(self, input_shape):
        if isinstance(input_shape, list):
            input_shape = input_shape[0]

        if not hasattr(self.cell, 'output_size') or not hasattr(self.cell, 'state_size'):
            raise ValueError(
                "RNN 셀은 출력 형태를 계산하기 위해 `output_size` 및 `state_size` 속성을 가져야 합니다."
                "셀이 올바르게 정의되었는지 확인하세요."
            )

        if self.return_sequences:
            # (batch_size, timesteps, output_size)
            output_shape = (input_shape[0], input_shape[1], self.cell.output_size)
        else:
            # (batch_size, output_size)
            output_shape = (input_shape[0], self.cell.output_size)

        if self.return_state:
            # 상태는 일반적으로 각 상태에 대해 (batch_size, state_size_i)입니다.
            state_shapes = [(input_shape[0], s) for s in (self.cell.state_size if isinstance(self.cell.state_size, list) else [self.cell.state_size])]
            return [output_shape] + state_shapes
        return output_shape

    def get_config(self):
        config = {
            "cell": tf.keras.utils.serialize_keras_object(self.cell),
            "return_sequences": self.return_sequences,
            "return_state": self.return_state,
            "go_backwards": self.go_backwards,
            "stateful": self.stateful,
            "unroll": self.unroll,
            "time_major": self.time_major,
        }
        base_config = super().get_config()
        return {**base_config, **config}

    @classmethod
    def from_config(cls, config):
        if "cell" in config:
            # 사용자 정의 셀의 경우 `tf.keras.utils.deserialize_keras_object` 및 `globals()` 액세스가 필요합니다.
            # 이 부분은 사용자 정의 컨텍스트에서 'cell'이 직렬화/역직렬화되는 방식에 따라 조정이 필요할 수 있습니다.
            cell_config = config.pop("cell")
            cell = tf.keras.layers.deserialize(cell_config) # 셀을 직접 역직렬화할 수 있다고 가정하여 단순화
            return cls(cell, **config)
        return cls(**config)

lstm

In [ ]:
from tensorflow.keras.layers import RNN
from tensorflow.keras import activations, initializers, regularizers, constraints

class LSTM(RNN):
    """Long Short-Term Memory 레이어 - Hochreiter 1997.

    사용 가능한 런타임 하드웨어 및 제약 조건에 따라 이 레이어는
    성능을 극대화하기 위해 다른 구현(cuDNN 기반 또는 백엔드 네이티브)을
    선택합니다. GPU를 사용할 수 있고 레이어에 대한 모든 인자가 cuDNN 커널의
    요구 사항을 충족하는 경우 (자세한 내용은 아래 참조), 이 레이어는
    TensorFlow 백엔드를 사용할 때 빠른 cuDNN 구현을 사용합니다.
    cuDNN 구현을 사용하기 위한 요구 사항은 다음과 같습니다:

    1. `activation` == `tanh`
    2. `recurrent_activation` == `sigmoid`
    3. `recurrent_dropout` == 0
    4. `unroll`이 `False`인 경우
    5. `use_bias`가 `True`인 경우
    6. 입력이 마스킹을 사용하는 경우 엄격하게 오른쪽으로 채워져야 합니다.
    7. 가장 바깥쪽 컨텍스트에서 Eager 실행이 활성화되어야 합니다.

    예시:

    >>> inputs = np.random.random((32, 10, 8))
    >>> lstm = keras.layers.LSTM(4)
    >>> output = lstm(inputs)
    >>> output.shape
    (32, 4)
    >>> lstm = keras.layers.LSTM(
    ...     4, return_sequences=True, return_state=True)
    >>> whole_seq_output, final_memory_state, final_carry_state = lstm(inputs)
    >>> whole_seq_output.shape
    (32, 10, 4)
    >>> final_memory_state.shape
    (32, 4)
    >>> final_carry_state.shape
    (32, 4)

    인자:
        units: 양의 정수, 출력 공간의 차원입니다.
        activation: 사용할 활성화 함수입니다.
            기본값: 하이퍼볼릭 탄젠트 (`tanh`).
            `None`을 전달하면 활성화가 적용되지 않습니다
            (즉, "선형" 활성화: `a(x) = x`).
        recurrent_activation: 순환 단계에 사용할 활성화 함수입니다.
            기본값: 시그모이드 (`sigmoid`).
            `None`을 전달하면 활성화가 적용되지 않습니다
            (즉, "선형" 활성화: `a(x) = x`).
        use_bias: 불리언, (기본값 `True`), 레이어가 바이어스 벡터를 사용해야 하는지 여부입니다.
        kernel_initializer: `kernel` 가중치 행렬의 초기화기입니다.
            입력의 선형 변환에 사용됩니다. 기본값:
            `"glorot_uniform"`.
        recurrent_initializer: `recurrent_kernel` 가중치 행렬의 초기화기입니다.
            순환 상태의 선형 변환에 사용됩니다. 기본값: `"orthogonal"`.
        bias_initializer: `bias` 벡터의 초기화기입니다. 기본값: `"zeros"`.
            `unit_forget_bias`: 불리언 (기본값 `True`). True인 경우, 초기화 시 잊각 게이트의 바이어스에 1을 더합니다. True로 설정하면 `len(units)`에서 상수만큼 감소된 값이 더해지는 연산이 발생하여, 훈련 초기에 셀이 모든 과거 경험을 잊는 것을 방지할 수 있습니다. 초기화 시 잊각 게이트 바이어스에 1을 더하는 것이 기울기 소실 문제를 방지하는 데 도움이 된다는 것이 입증되었습니다.
        kernel_regularizer: `kernel` 가중치 행렬에 적용되는 정규화 함수입니다.
        recurrent_regularizer: `recurrent_kernel` 가중치 행렬에 적용되는 정규화 함수입니다.
        bias_regularizer: `bias` 벡터에 적용되는 정규화 함수입니다.
        activity_regularizer: 레이어의 출력("활성화")에 적용되는 정규화 함수입니다.
        kernel_constraint: `kernel` 가중치 행렬에 적용되는 제약 함수입니다.
        recurrent_constraint: `recurrent_kernel` 가중치 행렬에 적용되는 제약 함수입니다.
        bias_constraint: `bias` 벡터에 적용되는 제약 함수입니다.
        dropout: 0과 1 사이의 부동 소수점 값입니다. 입력의 선형 변환을 위해 드롭할 단위의 비율입니다.
        recurrent_dropout: 0과 1 사이의 부동 소수점 값입니다. 순환 상태의 선형 변환을 위해 드롭할 단위의 비율입니다.
        return_sequences: 불리언. 출력 시퀀스의 마지막 출력을 반환할지, 아니면 전체 시퀀스를 반환할지 여부입니다. 기본값: `False`.
        return_state: 불리언. 출력 외에 마지막 상태를 반환할지 여부입니다. 기본값: `False`.
        go_backwards: 불리언 (기본값 `False`). True인 경우, 입력 시퀀스를 역방향으로 처리하고 역순 시퀀스를 반환합니다.
        stateful: 불리언 (기본값 `False`). True인 경우, 배치 내 인덱스 `i`의 각 샘플에 대한 마지막 상태는 다음 배치 내 인덱스 `i`의 샘플에 대한 초기 상태로 사용됩니다.
        unroll: 불리언 (기본값 `False`). True인 경우, 네트워크는 펼쳐지며, 그렇지 않으면 기호 루프가 사용됩니다. `input_shape`가 정적이고 작을 때 펼치기는 RNN의 속도를 높일 수 있습니다.
        time_major: 불리언 (기본값 `False`). True인 경우, 입력 및 출력은 `(timesteps, batch, feature)` 형태가 되며, `False` (기본값) 옵션은 `(batch, timesteps, feature)`를 사용합니다.
            `reset_after`: 불리언 (기본값 `True`). True인 경우, 각 셀 후에 순환 연결이 재설정됩니다. 구현 차이로 인해 CuDNN 백엔드의 경우 기본값은 `True`이고 Tensorflow 백엔드의 경우 `False`입니다.
    """

    def __init__(
        self,
        units,
        activation="tanh",
        recurrent_activation="sigmoid",
        use_bias=True,
        kernel_initializer="glorot_uniform",
        recurrent_initializer="orthogonal",
        bias_initializer="zeros",
        unit_forget_bias=True,
        kernel_regularizer=None,
        recurrent_regularizer=None,
        bias_regularizer=None,
        activity_regularizer=None,
        kernel_constraint=None,
        recurrent_constraint=None,
        bias_constraint=None,
        dropout=0.0,
        recurrent_dropout=0.0,
        return_sequences=False,
        return_state=False,
        go_backwards=False,
        stateful=False,
        unroll=False,
        time_major=False,
        reset_after=True,
        **kwargs,
    ):
        super().__init__(
            units,
            activation=activation,
            recurrent_activation=recurrent_activation,
            use_bias=use_bias,
            kernel_initializer=kernel_initializer,
            recurrent_initializer=recurrent_initializer,
            bias_initializer=bias_initializer,
            unit_forget_bias=unit_forget_bias,
            kernel_regularizer=kernel_regularizer,
            recurrent_regularizer=recurrent_regularizer,
            bias_regularizer=bias_regularizer,
            activity_regularizer=activity_regularizer,
            kernel_constraint=kernel_constraint,
            recurrent_constraint=recurrent_constraint,
            bias_constraint=bias_constraint,
            dropout=dropout,
            recurrent_dropout=recurrent_dropout,
            return_sequences=return_sequences,
            return_state=return_state,
            go_backwards=go_backwards,
            stateful=stateful,
            unroll=unroll,
            time_major=time_major,
            reset_after=reset_after,
            **kwargs,
        )

    def call(self, inputs, initial_state=None, training=None):
        return super().call(inputs, initial_state=initial_state, training=training)

    @property
    def units(self):
        return self.cell.units

    @property
    def activation(self):
        return self.cell.activation

    @property
    def recurrent_activation(self):
        return self.cell.recurrent_activation

    @property
    def use_bias(self):
        return self.cell.use_bias

    @property
    def kernel_initializer(self):
        return self.cell.kernel_initializer

    @property
    def recurrent_initializer(self):
        return self.cell.recurrent_initializer

    @property
    def bias_initializer(self):
        return self.cell.bias_initializer

    @property
    def unit_forget_bias(self):
        return self.cell.unit_forget_bias

    @property
    def kernel_regularizer(self):
        return self.cell.kernel_regularizer

    @property
    def recurrent_regularizer(self):
        return self.cell.recurrent_regularizer

    @property
    def bias_regularizer(self):
        return self.cell.bias_regularizer

    @property
    def activity_regularizer(self):
        return self.cell.activity_regularizer

    @property
    def kernel_constraint(self):
        return self.cell.kernel_constraint

    @property
    def recurrent_constraint(self):
        return self.cell.recurrent_constraint

    @property
    def bias_constraint(self):
        return self.cell.bias_constraint

    @property
    def dropout(self):
        return self.cell.dropout

    @property
    def recurrent_dropout(self):
        return self.cell.recurrent_dropout

    def get_config(self):
        config = {
            "units": self.units,
            "activation": activations.serialize(self.activation),
            "recurrent_activation": activations.serialize(
                self.recurrent_activation
            ),
            "use_bias": self.use_bias,
            "kernel_initializer": initializers.serialize(
                self.kernel_initializer
            ),
            "recurrent_initializer": initializers.serialize(
                self.recurrent_initializer
            ),
            "bias_initializer": initializers.serialize(self.bias_initializer),
            "unit_forget_bias": self.unit_forget_bias,
            "kernel_regularizer": regularizers.serialize(
                self.kernel_regularizer
            ),
            "recurrent_regularizer": regularizers.serialize(
                self.recurrent_regularizer
            ),
            "bias_regularizer": regularizers.serialize(self.bias_regularizer),
            "activity_regularizer": regularizers.serialize(
                self.activity_regularizer
            ),
            "kernel_constraint": constraints.serialize(self.kernel_constraint),
            "recurrent_constraint": constraints.serialize(
                self.recurrent_constraint
            ),
            "bias_constraint": constraints.serialize(self.bias_constraint),
            "dropout": self.dropout,
            "recurrent_dropout": self.recurrent_dropout,
            "seed": self.cell.seed,
        }
        base_config = super().get_config()
        del base_config["cell"]
        return {**base_config, **config}

    @classmethod
    def from_config(cls, config):
        return cls(**config)

## Transformer 모델 소스 코드





Transformer 모델은 주로 두 가지 핵심 구성 요소로 이루어져 있습니다: `MultiHeadSelfAttention`과 `TransformerBlock`. 이들을 구현하려면 `tensorflow` 및 `tensorflow.keras.layers`의 여러 모듈이 필요합니다. 먼저 `MultiHeadSelfAttention` 레이어부터 구현해 보겠습니다.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Dropout
from tensorflow.keras.models import Model

class MultiHeadSelfAttention(Layer):
    def __init__(self, embed_dim, num_heads=8):
        super(MultiHeadSelfAttention, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        if embed_dim % num_heads != 0:
            raise ValueError(
                f"임베딩 차원 = {embed_dim}은 헤드 수 = {num_heads}로 나누어 떨어져야 합니다."
            )
        self.projection_dim = embed_dim // num_heads
        self.query_dense = Dense(embed_dim)
        self.key_dense = Dense(embed_dim)
        self.value_dense = Dense(embed_dim)
        self.combine_heads = Dense(embed_dim)

    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output, weights

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]

        query = self.query_dense(inputs)  # (batch_size, seq_len, embed_dim)
        key = self.key_dense(inputs)  # (batch_size, seq_len, embed_dim)
        value = self.value_dense(inputs)  # (batch_size, seq_len, embed_dim)

        query = self.separate_heads(
            query, batch_size
        )  # (batch_size, num_heads, seq_len, projection_dim)
        key = self.separate_heads(
            key, batch_size
        )  # (batch_size, num_heads, seq_len, projection_dim)
        value = self.separate_heads(
            value, batch_size
        )  # (batch_size, num_heads, seq_len, projection_dim)

        attention, weights = self.attention(query, key, value)

        attention = tf.transpose(
            attention, perm=[0, 2, 1, 3]
        )  # (batch_size, seq_len, num_heads, projection_dim)
        concat_attention = tf.reshape(
            attention, (batch_size, -1, self.embed_dim)
        )  # (batch_size, seq_len, embed_dim)

        output = self.combine_heads(
            concat_attention
        )  # (batch_size, seq_len, embed_dim)
        return output

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
        })
        return config

다음으로, `MultiHeadSelfAttention` 레이어와 피드포워드 네트워크, 정규화 및 드롭아웃을 포함하는 `TransformerBlock`을 구현하겠습니다. 이는 Transformer 인코더의 핵심 블록입니다.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Dropout, LayerNormalization

class TransformerBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential(
            [
                Dense(ff_dim, activation="relu"),
                Dense(embed_dim),
            ]
        )
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate

    def call(self, inputs, training):
        attn_output = self.att(inputs)  # 셀프 어텐션 출력
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)  # 추가 및 정규화
        ffn_output = self.ffn(out1)  # 피드포워드 네트워크 출력
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)  # 추가 및 정규화

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "rate": self.rate,
        })
        return config

마지막으로, 입력 임베딩과 위치 인코딩을 처리하고 여러 `TransformerBlock`을 쌓아

완전한 Transformer 인코더 모델을 구성합니다.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Embedding, Input
from tensorflow.keras.models import Model

class TokenAndPositionEmbedding(Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = Embedding(input_dim=maxlen, output_dim=embed_dim)
        self.maxlen = maxlen
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

    def get_config(self):
        config = super().get_config()
        config.update({
            "maxlen": self.maxlen,
            "vocab_size": self.vocab_size,
            "embed_dim": self.embed_dim,
        })
        return config

# 간단한 Transformer 인코더 모델로 결합하는 방법의 예시입니다:
# 이는 전체 모델에 대한 직접적인 클래스 정의가 아닌, 함수형 API 예시입니다.

def create_transformer_encoder_model(maxlen, vocab_size, embed_dim, num_heads, ff_dim, num_transformer_blocks=2, dropout_rate=0.1):
    inputs = Input(shape=(maxlen,))
    embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
    x = embedding_layer(inputs)

    for _ in range(num_transformer_blocks):
        transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim, rate=dropout_rate)
        x = transformer_block(x)

    x = LayerNormalization(epsilon=1e-6)(x) # 선택적 최종 정규화
    outputs = Dense(vocab_size, activation="softmax")(x) # 분류를 위한 예시 출력 레이어

    model = Model(inputs=inputs, outputs=outputs)
    return model

# 이 예시 모델을 사용하려면:
# vocab_size = 20000  # 상위 2만 개 단어만 고려
# maxlen = 200  # 최대 시퀀스 크기
# embed_dim = 32  # 각 토큰에 대한 임베딩 크기
# num_heads = 2  # 어텐션 헤드 수
# ff_dim = 32  # 트랜스포머 내부 피드포워드 네트워크의 숨겨진 레이어 크기

# transformer_model = create_transformer_encoder_model(maxlen, vocab_size, embed_dim, num_heads, ff_dim)
# transformer_model.summary()